In [1]:
%pip install pandas numpy statsmodels


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split

# loading csv
df = pd.read_csv("../data/train.csv")



C:\Users\perez\AppData\Local\Temp\ipykernel_6704\352076619.py:8: DtypeWarning: Columns (11: Grid) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/train.csv")


In [3]:
# cleaning whitespace from text columns
cols = [
    "Priority",
    "InitialCaseTypeGroup",
    "CallReceivedType",
    "OccurrenceHour",
    "OccurrenceWeekday",
    "OccurrenceMonth",
    "Zone",
]
df_clean = df.dropna(subset=cols).copy()

# dropping rare categories to prevent errors we can add them in later if yall think we need them and deal with issues
for col in ["InitialCaseTypeGroup", "CallReceivedType", "Zone"]:
    counts = df_clean[col].value_counts()
    valid_categories = counts[counts >= 15].index
    df_clean = df_clean[df_clean[col].isin(valid_categories)]



In [ ]:
# Priority as categorical for multinomial regression

df_clean["Priority"] = pd.Categorical(df_clean["Priority"])

print(f"Total rows after cleaning: {len(df_clean):,}")

# Train/Test Split ( 80% train, 20% test)
train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42)



Total rows after cleaning: 386,372


In [5]:
# turn priotity into integer
train_df["Priority"] = train_df["Priority"].astype(int)
test_df["Priority"] = test_df["Priority"].astype(int)

# Fitting Multinomial Logistic Regression Model
# Priority is now the dependent variable
formula = """
Priority ~ C(InitialCaseTypeGroup) 
         + C(CallReceivedType) 
         + OccurrenceHour 
         + C(OccurrenceWeekday) 
         + C(OccurrenceMonth) 
         + C(Zone)
"""

model = smf.mnlogit(formula, data=train_df).fit(method='bfgs', maxiter=3000, disp=True)

#Predict on the test set and output to a test file
# `model.predict()` returns a DataFrame of probabilities for each priority category

pred_probs = model.predict(test_df)
test_df["predicted_priority"] = pred_probs.idxmax(axis=1)

# statistical summary
print()
print(model.summary())



Optimization terminated successfully.
         Current function value: 0.777873
         Iterations: 1147
         Function evaluations: 1152
         Gradient evaluations: 1152

                          MNLogit Regression Results                          
Dep. Variable:               Priority   No. Observations:               309097
Model:                        MNLogit   Df Residuals:                   308779
Method:                           MLE   Df Model:                          312
Date:                Wed, 25 Nov 2026   Pseudo R-squ.:                  0.2834
Time:                        18:02:33   Log-Likelihood:            -2.4044e+05
converged:                       True   LL-Null:                   -3.3555e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                               Priority=2       coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------

In [6]:

#save to csv
test_df.to_csv("test_predictions.csv", index=False)
